In [7]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    "tokenizer/vocab.json",
    "tokenizer/merges.txt"
)
text = "Hello, world!"
encoded = tokenizer.encode(text)

print(encoded.ids)
print(encoded.tokens)
print(tokenizer.decode(encoded.ids))

[44, 5626, 16, 1582, 5]
['H', 'ello', ',', 'Ġworld', '!']
Hello, world!


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "final_model/step_6000.pt"

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        batch_size, sequence_length, n_embd = x.shape
        q, k, v = self.qkv(x).split(n_embd, dim=2)
        q = q.view(batch_size, sequence_length, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, sequence_length, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, sequence_length, self.n_head, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        scores = scores.masked_fill(self.causal_mask[:, :, :sequence_length, :sequence_length] == 0, float("-inf"))
        weights = self.dropout(F.softmax(scores, dim=-1))
        output = (weights @ v).transpose(1, 2).contiguous().view(batch_size, sequence_length, n_embd)
        return self.dropout(self.projection(output))

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(n_embd)
        self.attention = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.layer_norm_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attention(self.layer_norm_1(x))
        return x + self.mlp(self.layer_norm_2(x))

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.block_size = config["block_size"]
        self.token_embedding = nn.Embedding(config["vocab_size"], config["n_embd"])
        self.position_embedding = nn.Embedding(config["block_size"], config["n_embd"])
        self.blocks = nn.Sequential(*[TransformerBlock(config["n_embd"], config["n_head"], config["block_size"], config["dropout"]) for _ in range(config["n_layer"])])
        self.final_layer_norm = nn.LayerNorm(config["n_embd"])
        self.language_model_head = nn.Linear(config["n_embd"], config["vocab_size"], bias=False)
        self.language_model_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        sequence_length = input_ids.size(1)
        positions = torch.arange(sequence_length, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.final_layer_norm(self.blocks(x))
        return self.language_model_head(x)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model = GPT(checkpoint["model_config"]).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from step {checkpoint['step']} on {DEVICE}")

Loaded checkpoint from step 6000 on cpu


In [12]:
def generate(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    encoded = tokenizer.encode(prompt)
    input_ids = torch.tensor([encoded.ids], dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = input_ids[:, -model.block_size:]
            logits = model(context)[:, -1, :] / temperature

            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < values[:, [-1]]] = float("-inf")
            probabilities = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            input_ids = torch.cat((input_ids, next_token), dim=1)

    return tokenizer.decode(input_ids[0].tolist())

prompt = "Cristiano Ronaldo is a "
print(generate(prompt, max_new_tokens=256, temperature=0.8, top_k=50))

Cristiano Ronaldo is a  American football offensive midfielder.

Background
On 22 March, she was the only senior manager of the Minnesota Vikings in the summer of 1963. At the age of 21 he was appointed for the first time in 1965. He was appointed mayor of the Minnesota Vikings in 1974, and managed as a headmaster of the Minnesota Vikings from 1977 to 1986.

Career
He was a member of the Minnesota Twins in 1996, and was appointed interim Director of the Minnesota Oilers.

Early years
Born in the Cleveland Browns, Cincinnati, Ohio from 1982 to 1982, he served on the National Football Hall of Fame as a member of the Colorado Vikings, Ohio Steelers and the Colorado Oilers. In 1981 he also played a National League Baseball All-Star team at Lincoln Stadium (NBA) in Denver, and Toronto.

He made his four-time career at the Cincinnati Reds during the 1981 season, but also played in the Houston Astros.

In 1987, he finished his first season for the Pittsburgh Jaguars. During his first season, 